In [ ]:
sentence = "오늘 날씨 어때?"   #sentence에 오늘 날씨 어때? 를 저장

# 공백을 기준으로 문장 나누기
tokens = sentence.split()     #sentence를 공백 기준으로 나누어 tokens에 저장

print(tokens)                 # tokens 출력

['오늘', '날씨', '어때?']


In [4]:
sentence = "오늘 날씨 어때?"
tokens = sentence.split()

# 각 토큰에 번호표를 붙임
vocabulary = {
    "오늘": 1,
    "날씨": 2,
    "어때?": 3,
}

token_ids = [vocabulary[token] for token in tokens]

print("토큰:", tokens)
print("토큰 ID:", token_ids)

토큰: ['오늘', '날씨', '어때?']
토큰 ID: [1, 2, 3]


In [9]:
token_ids = [1, 2, 3]

embedding_table = {                 #실제 학습값이 아닌 예시 
    1: [0.2, 0.5, 0.1],
    2: [0.7, 0.3, 0.4],
    3: [0.1, 0.8, 0.6],
}

embeddings = [embedding_table[token_id] for token_id in token_ids]

print(embeddings)

[[0.2, 0.5, 0.1], [0.7, 0.3, 0.4], [0.1, 0.8, 0.6]]


In [ ]:
#모델의 숫자 예측을 점점 정답에 가깝게 수정하는 과정

number = 0.0   # 수정할 숫자
answer = 2.0   # 정답

for i in range(3):
    error = answer - number   # 정답과 얼마나 차이 나는지
    number = number + error * 0.5  # 차이의 절반만큼 수정

    print(number)

1.0
1.5
1.75


In [ ]:
weight = 0.0       # 모델의 학습할 숫자
answer = 2.0       # 정답
learning_rate = 0.1  # 한 번에 수정할 크기

for i in range(3):
    prediction = weight * 1                         # 모델이 예측한 값

    loss = (prediction - answer) ** 2               #손실(loss) 계산

    gradient = 2 * (prediction - answer)            #손실(loss)을 줄이기 위해 weight를 얼마나 수정해야 하는지 계산

    weight = weight - learning_rate * gradient      #weight를 수정

    print("반복:", i + 1)
    print("수정된 weight:", round(weight, 3))
    print("수정 전 loss:", round(loss, 3))
    print()

반복: 1
수정된 weight: 0.4
수정 전 loss: 4.0

반복: 2
수정된 weight: 0.72
수정 전 loss: 2.56

반복: 3
수정된 weight: 0.976
수정 전 loss: 1.638



실제 모델과 연결하면
실제 모델에서는 사람이 gradient 식을 숫자마다 직접 적지 않고, PyTorch 같은 도구가 역전파를 통해 각 숫자의 기울기를 계산해요. 그 결과를 이용해 임베딩 테이블을 포함한 모델의 숫자들을 수정합니다.

지금은 아래 흐름을 이해하면 충분해요.
모델이 예상
    ↓
정답과 비교해서 loss 계산
    ↓
loss를 줄이려면 숫자를 어떻게 바꿔야 하는지 gradient 계산
    ↓
gradient를 이용해 숫자를 조금 수정
    ↓
반복

In [ ]:
#임베딩 안의 숫자가 학습하면서 바뀌는 모습을 보여줌

import torch
from torch import nn

# 예제를 다시 실행해도 같은 초기 숫자가 나오도록 설정
torch.manual_seed(42)

# 토큰 번호 1, 2, 3
token_ids = torch.tensor([1, 2, 3])

# 각 토큰에 대해 모델이 맞혀야 할 가상의 정답 번호
answers = torch.tensor([0, 1, 2])

# 1. 임베딩 테이블 만들기
# 토큰 번호 0~3을 각각 숫자 3개로 표현
embedding = nn.Embedding(4, 3)

# 2. 임베딩을 받아 정답 번호를 예측할 간단한 층
classifier = nn.Linear(3, 3)

# 3. 모델이 얼마나 틀렸는지 계산할 방법
loss_function = nn.CrossEntropyLoss()

# 4. 임베딩과 예측기의 숫자를 수정할 방법
optimizer = torch.optim.SGD(
    list(embedding.parameters()) + list(classifier.parameters()),
    lr=0.1
)

# 수정하기 전 1번 토큰의 임베딩을 따로 보관
before = embedding.weight[1].detach().clone()

# 기존에 남아 있는 기울기 초기화
optimizer.zero_grad()

# 토큰 ID로 임베딩을 찾음
vectors = embedding(token_ids)

# 임베딩을 이용해 정답 번호를 예측
predictions = classifier(vectors)


# 오차 확인 -> loss
# 어느 방향으로 고칠지 계산 -> gradient
#숫자를 실제로 수정 -> optimizer.step()

# 예측과 정답을 비교해 오차 계산
# 중요 코드1

# 모델이 예상한 답이 정답과 얼마나 다른지 확인
loss = loss_function(predictions, answers)

# 오차를 바탕으로 각 숫자를 어떻게 수정할지 계산
# 중요 코드2

# 덜 틀리려면 모델의 숫자들을 어떻게 바꿔야 하는지 계산
loss.backward()     #기울기 계산

# 1번 토큰의 임베딩에 대한 기울기 확인
gradient = embedding.weight.grad[1].clone()

# 계산한 기울기를 이용해 숫자 수정
# 중요 코드3

#계산한 방향대로 모델의 숫자들을 조금씩 수정
optimizer.step()    #기울기를 오차가 작아지는 방향으로 숫자들을 수정

# 수정된 1번 토큰의 임베딩
after = embedding.weight[1].detach()

print("수정 전:", before)
print("기울기:", gradient)
print("수정 후:", after)
print("오차:", loss.item())

수정 전: tensor([ 0.2303, -1.1229, -0.1863])
기울기: tensor([ 0.1824, -0.0225,  0.0870])
수정 후: tensor([ 0.2121, -1.1206, -0.1950])
오차: 1.7088769674301147


In [11]:
import torch

# 학습할 숫자
weight = torch.tensor(0.0, requires_grad=True)

# 정답
answer = 2.0

# 모델의 예상
prediction = weight * 1

# 예상과 정답의 차이 계산
loss = (prediction - answer) ** 2

# 기울기 계산
loss.backward()

print("예상:", prediction.item())
print("오차:", loss.item())
print("기울기:", weight.grad.item())

예상: 0.0
오차: 4.0
기울기: -4.0


In [1]:
import torch

# 1. 모델이 수정할 숫자
weight = torch.tensor(0.0, requires_grad=True)

# 2. 숫자를 수정할 도구
optimizer = torch.optim.SGD([weight], lr=0.1)

# 3. 예상값과 정답
prediction = weight * 1
answer = 2.0

# 4. 얼마나 틀렸는지 계산
loss = (prediction - answer) ** 2

# 5. 숫자를 수정할 방향 계산
loss.backward()

print("수정 전:", weight.item())
print("기울기:", weight.grad.item())

# 6. 숫자 수정
optimizer.step()

print("수정 후:", weight.item())

수정 전: 0.0
기울기: -4.0
수정 후: 0.4000000059604645
